In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import datetime
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_validate
from tqdm import tqdm

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/predicting-stock-trends-rise-or-fall/sample_submission.csv
/kaggle/input/predicting-stock-trends-rise-or-fall/train.csv
/kaggle/input/predicting-stock-trends-rise-or-fall/test.csv


# Load Data

In [2]:
data_path = dirname

In [3]:
df_train = pd.read_csv(f"{data_path}/train.csv", parse_dates=["Date"])
df_test = pd.read_csv(f"{data_path}/test.csv", parse_dates=["Date"])
df_submission = pd.read_csv(f"{data_path}/sample_submission.csv")

In [4]:
TRAIN_FROM = "2023-01-01"

In [5]:
df_train = df_train[df_train["Date"] >= TRAIN_FROM]

In [6]:
df_train.head()

,Ticker,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
18889344,ticker_681,2023-01-03,1.15,1.150,1.02,1.04,28300.0,0.0,0.0
18889345,ticker_1390,2023-01-03,23.57,25.000,23.57,24.99,900.0,0.0,0.0
18889346,ticker_1619,2023-01-03,5.80,6.130,5.76,5.85,201600.0,0.0,0.0
18889347,ticker_4561,2023-01-03,13.97,14.385,12.49,13.44,38900.0,0.0,0.0
18889348,ticker_4646,2023-01-03,10.90,10.900,10.90,10.90,0.0,0.0,0.0


In [7]:
df_train.tail()

,Ticker,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
21033517,ticker_2098,2024-09-23,15.230000,15.230000,14.690000,14.780000,63400.0,0.0,0.0
21033518,ticker_3748,2024-09-23,24.700001,24.768400,24.510000,24.570000,31208.0,0.0,0.0
21033519,ticker_2615,2024-09-23,11.090000,11.150000,10.960000,11.090000,1014300.0,0.0,0.0
21033520,ticker_4765,2024-09-23,25.200001,25.202999,25.129999,25.134001,4500.0,0.0,0.0
21033521,ticker_4658,2024-09-23,0.281000,0.290000,0.264000,0.270000,272200.0,0.0,0.0


In [8]:
df_train.describe()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
count,2144178,2.144178e+06,2.144178e+06,2.144178e+06,2.144178e+06,2.144178e+06,2144178.0,2144178.0
mean,2023-11-14 16:58:33.738131968,3.278992e+01,3.324300e+01,3.234071e+01,3.278963e+01,1.272846e+06,0.0,0.0
min,2023-01-03 00:00:00,1.000000e-02,7.900000e-02,1.000000e-02,7.851000e-02,0.000000e+00,0.0,0.0
25%,2023-06-12 00:00:00,6.280000e+00,6.440000e+00,6.120000e+00,6.270000e+00,4.519725e+04,0.0,0.0
50%,2023-11-14 00:00:00,1.575000e+01,1.600000e+01,1.550000e+01,1.575000e+01,2.331000e+05,0.0,0.0
75%,2024-04-19 00:00:00,3.592000e+01,3.645000e+01,3.537000e+01,3.591000e+01,9.056000e+05,0.0,0.0
max,2024-09-23 00:00:00,4.293400e+02,5.754100e+02,4.220000e+02,4.288500e+02,5.617574e+08,0.0,0.0
std,NaN,4.732431e+01,4.787046e+01,4.678600e+01,4.733780e+01,4.876551e+06,0.0,0.0


In [9]:
df_train[df_train["Stock Splits"] != 0.0]

,Ticker,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits


# Feature Engineering

In [10]:
# Create Label: Get the closing price after 30 business days for each stock, 1 if up, 0 if down
df_train = df_train.sort_values(["Ticker", "Date"])
df_train["future_close"] = df_train.groupby("Ticker")["Close"].shift(-30)
df_train["target"] = (df_train["future_close"] > df_train["Close"]).astype(int)

In [11]:
def create_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["Ticker", "Date"])
    # Daily return
    df["ret_1"] = df.groupby("Ticker")["Close"].pct_change(1)
    # Moving average (5, 10, 20 days)
    for w in [5, 10, 20]:
        df[f"ma_{w}"] = df.groupby("Ticker")["Close"].transform(lambda x: x.rolling(w).mean())
    # volatility (5,10,20日)
    for w in [5,10,20]:
        df[f'vol_{w}'] = df.groupby('Ticker')['ret_1'].transform(lambda x: x.rolling(w).std())
    return df

In [12]:
df_train = create_features(df_train)

In [13]:
df_train.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Ticker,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,future_close,target,ret_1,ma_5,ma_10,ma_20,vol_5,vol_10,vol_20
18890437,ticker_1,2023-01-03,95.459999,96.370003,93.930000,95.760002,2098400.0,0.0,0.0,91.199997,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18896601,ticker_1,2023-01-04,96.239998,97.440002,95.940002,96.589996,1878500.0,0.0,0.0,91.660004,0,0.008667,NaN,NaN,NaN,NaN,NaN,NaN
18903105,ticker_1,2023-01-05,95.889999,96.379997,94.430000,95.290001,1891600.0,0.0,0.0,93.250000,0,-0.013459,NaN,NaN,NaN,NaN,NaN,NaN
18906387,ticker_1,2023-01-06,96.269997,98.070000,96.019997,97.830002,1414500.0,0.0,0.0,92.730003,0,0.026655,NaN,NaN,NaN,NaN,NaN,NaN
18910803,ticker_1,2023-01-09,96.809998,98.650002,96.290001,97.720001,1471300.0,0.0,0.0,92.410004,0,-0.001124,96.638,NaN,NaN,NaN,NaN,NaN


In [14]:
df_train.tail()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Ticker,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,future_close,target,ret_1,ma_5,ma_10,ma_20,vol_5,vol_10,vol_20
21010467,ticker_999,2024-09-17,87.059998,88.120003,87.059998,87.709999,997900.0,0.0,0.0,NaN,0,0.009553,85.925998,84.388999,84.756499,0.005079,0.007838,0.017540
21014125,ticker_999,2024-09-18,87.720001,87.959999,86.680000,86.870003,757100.0,0.0,0.0,NaN,0,-0.009577,86.429999,84.770999,84.766999,0.009428,0.009242,0.017685
21019828,ticker_999,2024-09-19,88.220001,89.089996,87.680000,88.800003,885900.0,0.0,0.0,NaN,0,0.022217,87.164000,85.353999,84.873999,0.011999,0.010525,0.018352
21026462,ticker_999,2024-09-20,89.019997,90.070000,88.610001,89.910004,1665400.0,0.0,0.0,NaN,0,0.012500,88.034001,86.112000,85.031000,0.011921,0.009282,0.018520
21032385,ticker_999,2024-09-23,90.070000,90.239998,89.480003,89.989998,569600.0,0.0,0.0,NaN,0,0.000890,88.656001,86.809000,85.116500,0.012046,0.009623,0.018159


In [15]:
# Columns used for training
feature_cols = ['ret_1'] + [f'ma_{w}' for w in [5,10,20]] + [f'vol_{w}' for w in [5,10,20]]

In [16]:
feature_cols

['ret_1', 'ma_5', 'ma_10', 'ma_20', 'vol_5', 'vol_10', 'vol_20']

In [17]:
# Extract data for training (only rows with complete labels and features)
train_model = df_train.dropna(subset=['target'] + feature_cols)
X = train_model[feature_cols]
y = train_model['target']

In [18]:
# Split into training/validation data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [19]:
# Train Model
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    eval_metric='logloss'
)

In [20]:
model.fit(
    X_train, y_train,
    eval_set=[(X_train,y_train),(X_val,y_val)],
    verbose=True
)

[0]	validation_0-logloss:0.68806	validation_1-logloss:0.68802
[1]	validation_0-logloss:0.68699	validation_1-logloss:0.68692
[2]	validation_0-logloss:0.68611	validation_1-logloss:0.68602
[3]	validation_0-logloss:0.68537	validation_1-logloss:0.68525
[4]	validation_0-logloss:0.68476	validation_1-logloss:0.68463
[5]	validation_0-logloss:0.68422	validation_1-logloss:0.68407
[6]	validation_0-logloss:0.68377	validation_1-logloss:0.68361
[7]	validation_0-logloss:0.68340	validation_1-logloss:0.68322
[8]	validation_0-logloss:0.68307	validation_1-logloss:0.68289
[9]	validation_0-logloss:0.68278	validation_1-logloss:0.68259
[10]	validation_0-logloss:0.68251	validation_1-logloss:0.68231
[11]	validation_0-logloss:0.68232	validation_1-logloss:0.68211
[12]	validation_0-logloss:0.68212	validation_1-logloss:0.68191
[13]	validation_0-logloss:0.68196	validation_1-logloss:0.68175
[14]	validation_0-logloss:0.68180	validation_1-logloss:0.68158
[15]	validation_0-logloss:0.68166	validation_1-logloss:0.68144
[1

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, random_state=None, ...)

In [21]:
# Create features & predictions on test data
# Combine train data and test brand list and generate features with the same function
df_all = pd.concat([
    df_train[['Ticker','Date','Close']],
    df_test.rename(columns={'ID':'Ticker'})[['Ticker','Date']]
], ignore_index=True, sort=False)
df_all = create_features(df_all)

/tmp/ipykernel_13/4260301780.py:4: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["ret_1"] = df.groupby("Ticker")["Close"].pct_change(1)


In [22]:
df_all.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,Ticker,Date,Close,ret_1,ma_5,ma_10,ma_20,vol_5,vol_10,vol_20
0,ticker_1,2023-01-03,95.760002,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ticker_1,2023-01-04,96.589996,0.008667,NaN,NaN,NaN,NaN,NaN,NaN
2,ticker_1,2023-01-05,95.290001,-0.013459,NaN,NaN,NaN,NaN,NaN,NaN
3,ticker_1,2023-01-06,97.830002,0.026655,NaN,NaN,NaN,NaN,NaN,NaN
4,ticker_1,2023-01-09,97.720001,-0.001124,96.638,NaN,NaN,NaN,NaN,NaN


In [23]:
# Extract only features for the forecast target date (2024-11-04)
test_feats = df_all[df_all['Date']==pd.to_datetime('2024-11-04')]
X_test = test_feats.set_index('Ticker')[feature_cols]

In [24]:
# Predict
preds = model.predict(X_test)

In [25]:
# Make Submission File
submission = df_test.copy()
submission['Pred'] = submission['ID'].map(dict(zip(X_test.index, preds)))

In [26]:
submission.describe()

,Date,Pred
count,5000,5000.000000
mean,2024-11-04 00:00:00,0.070600
min,2024-11-04 00:00:00,0.000000
25%,2024-11-04 00:00:00,0.000000
50%,2024-11-04 00:00:00,0.000000
75%,2024-11-04 00:00:00,0.000000
max,2024-11-04 00:00:00,1.000000
std,NaN,0.256181


In [27]:
submission.head()

,ID,Date,Pred
0,ticker_1,2024-11-04,0
1,ticker_10,2024-11-04,0
2,ticker_100,2024-11-04,0
3,ticker_1000,2024-11-04,0
4,ticker_1001,2024-11-04,0


In [28]:
submission.drop(["Date"], axis=1).to_csv('submission.csv', index=False)

# Cross Validation

In [29]:
# TimeSeriesSplit with 5 divisions
tscv = TimeSeriesSplit(n_splits=5)

In [30]:
# When multiple evaluation indicators are calculated at once
cv_results = cross_validate(
    model,
    X,
    y,
    cv=tscv,
    scoring=['accuracy','roc_auc'],
    return_train_score=True
)

In [31]:
print("Accuracy for each division:", cv_results['test_accuracy'])
print("Average accuracy:", cv_results['test_accuracy'].mean())
print("AUC for each division:", cv_results['test_roc_auc'])
print("Average AUC:", cv_results['test_roc_auc'].mean())

Accuracy for each division: [0.5389919  0.54501774 0.56108471 0.56993416 0.56396996]
Average accuracy: 0.5557996941581865
AUC for each division: [0.54255139 0.54312973 0.5520455  0.53865723 0.57936904]
Average AUC: 0.5511505764715333
